### Testing SUMO accessibility via python code.

In [12]:
import os
import sys
import traci

# Check SUMO_HOME
if 'SUMO_HOME' in os.environ:
    tools = os.path.join(os.environ['SUMO_HOME'], 'tools')
    sys.path.append(tools)
else:
    sys.exit("Please declare environment variable 'SUMO_HOME'")

# Choose sumo or sumo-gui
sumoBinary = "sumo-gui"   # use "sumo" for command line

sumoCmd = [sumoBinary, "-c", "../nets/unrestricted_right_xml_gen/unrestricted_right.sumocfg", "--delay=50", "--start", "--quit-on-end"]

# Start SUMO
traci.start(sumoCmd)

step = 0
while step < 3600:
    traci.simulationStep()
    step += 1
    
    # vehicle_ids = traci.vehicle.getIDList()
    # print("Vehicles in simulation:", vehicle_ids)

    # print(traci.lane.getLastStepHaltingNumber("westJunction_1"))  # Example: get queue length for a specific lane

traci.close()

In [11]:
traci.close()

#### Setting up SUMO RL environment for running simulation

In [ ]:
# Paths for SUMO configuration and network files
import os

net_file = "../nets/unrestricted_right_xml_gen/unrestricted_right.net.xml"
route_file = "../nets/unrestricted_right_xml_gen/unrestricted_right.rou.xml"
output_file = "../outputs/sumo_rl_outputs.csv"
# sumocfg_file = "../nets/unrestricted_right_xml_gen/unrestricted_right.sumocfg"

# print("Net file path:", net_file,"Route file path:", route_file, "Output file path:", output_file, sep="\n")


Net file path:
../nets/unrestricted_right_xml_gen/unrestricted_right.net.xml
Route file path:
../nets/unrestricted_right_xml_gen/unrestricted_right.rou.xml
Output file path:
../outputs/sumo_rl_outputs.csv


In [ ]:
import os
import numpy as np
from pathlib import Path
from gymnasium import spaces
from sumo_rl import SumoEnvironment
from sumo_rl.environment.traffic_signal import TrafficSignal
from sumo_rl.environment.observations import ObservationFunction



# Reward Function
# In single-agent mode, sumo-rl still calls this per TrafficSignal,
# but there is only ONE signal, so no filtering logic is needed.
def reward_function(traffic_signal: TrafficSignal) -> float:
    # Waiting time
    waiting_time_raw = traffic_signal.get_accumulated_waiting_time_per_lane()
    
    # get_accumulated_waiting_time_per_lane() returns a dict {lane_id: float}
    # sum the values to get a single float
    if isinstance(waiting_time_raw, dict):
        total_waiting = sum(waiting_time_raw.values())
    elif isinstance(waiting_time_raw, (int, float)):
        total_waiting = float(waiting_time_raw)
    else:
        total_waiting = float(sum(waiting_time_raw))  # handles any other iterable

    #Queue length 
    # getLastStepHaltingNumber returns int per lane — sum across all lanes
    queue_length = float(sum(
        traffic_signal.sumo.lane.getLastStepHaltingNumber(lane)
        for lane in traffic_signal.lanes
    ))

    #  Pressure
    # get_pressure() should return float — cast defensively
    pressure = float(traffic_signal.get_pressure())

    # Collisions
    collision_count = float(len(traffic_signal.sumo.simulation.getCollisions()))

    # # ── Debug print (remove once working) ─────────────────────────────────────
    # print(
    #     f"total_waiting={total_waiting}({type(total_waiting).__name__}) | "
    #     f"queue={queue_length}({type(queue_length).__name__}) | "
    #     f"pressure={pressure}({type(pressure).__name__}) | "
    #     f"collisions={collision_count}({type(collision_count).__name__})"
    # )

    # ── Reward ─────────────────────────────────────────────────────────────────
    reward = -(
        total_waiting   * 0.1 +
        queue_length    * 0.5 +
        abs(pressure)   * 0.3 +
        collision_count * 5.0
    )

    return reward

# Observation Function

class CustomObservationFunction(ObservationFunction):
    """
    Observation vector:
    [phase_one_hot | density_per_lane | queue_per_lane | speed_per_lane]
    """

    def __init__(self, ts: TrafficSignal):
        super().__init__(ts)

    def __call__(self) -> np.ndarray:
        ts = self.ts

        # One-hot encode current green phase
        phase_id = [1 if ts.green_phase == i else 0 for i in range(ts.num_green_phases)]

        # Per-lane density (vehicles / lane capacity)
        density = [
            ts.sumo.lane.getLastStepVehicleNumber(lane) / (ts.sumo.lane.getLength(lane) / 7.5)
            for lane in ts.lanes
        ]

        # Per-lane queue ratio
        queue = [
            ts.sumo.lane.getLastStepHaltingNumber(lane) / (ts.sumo.lane.getLength(lane) / 7.5)
            for lane in ts.lanes
        ]

        # Per-lane mean speed normalised by speed limit
        speed = [
            ts.sumo.lane.getLastStepMeanSpeed(lane) /
            max(ts.sumo.lane.getMaxSpeed(lane), 1e-6)
            for lane in ts.lanes
        ]

        obs = np.array(phase_id + density + queue + speed, dtype=np.float32)
        return np.clip(obs, 0.0, 1.0)

    def observation_space(self) -> spaces.Box:
        ts = self.ts
        n_lanes = len(ts.lanes)
        size = ts.num_green_phases + 3 * n_lanes
        return spaces.Box(low=0.0, high=1.0, shape=(size,), dtype=np.float32)


# ─────────────────────────────────────────────
# Environment Factory
# ─────────────────────────────────────────────
class EnvGenerator:
    def __init__(
        self,
        net_file: str,
        route_file: str,
        out_csv_name: str,
        duration: int,
        use_gui: bool = False,
    ):
        self.net_file = net_file
        self.route_file = route_file
        self.out_csv_name = out_csv_name
        self.duration = duration
        self.use_gui = use_gui

    def get_training_environment(self) -> SumoEnvironment:
        env = SumoEnvironment(
            net_file=self.net_file,
            route_file=self.route_file,
            out_csv_name=self.out_csv_name,
            single_agent=True,           # ← Gymnasium interface, one signal
            use_gui=self.use_gui,
            num_seconds=self.duration,
            delta_time=5,
            yellow_time=2,
            min_green=5,
            reward_fn=reward_function,
            observation_class=CustomObservationFunction,
        )
        return env


# Smoke-test
if __name__ == "__main__":
    # base = Path(__file__).resolve().parent.parent
    folder_name = "unrestricted_right_xml_gen"
    net_name = "unrestricted_right"

    net_file     = Path().joinpath("..", "nets",  folder_name, f"{net_name}.net.xml")
    route_file    = Path().joinpath("..", "nets",  folder_name, "routes.rou.xml")
    out_csv_name    = Path().joinpath("..", "outputs",  "sumo_rl_outputs")
    # route_file   = str(".." / "nets" / "unrestricted_right_xml_gen" / "unrestricted_right.rou.xml")
    # out_csv_name = str(".." / "outputs" / "sumo_rl_outputs")

    # os.makedirs(Path().joinpath("..", "outputs", exist_ok=True))
    Path("..", "outputs").mkdir(parents=True, exist_ok=True)

    env = EnvGenerator(net_file, route_file, out_csv_name, duration=3600).get_training_environment()

    # Single-agent Gymnasium reset 
    # Returns (obs_array, info) — NOT a dict like PettingZoo
    obs, info = env.reset()
    print(f"✓ Reset  | obs shape : {obs.shape}  | obs: {obs}")

    # Single-agent step 
    # action  → single int, NOT a dict
    # returns → (obs, reward, terminated, truncated, info)
    print("\nRunning 5 random steps ...")
    for step in range(5):
        action = env.action_space.sample()          # single int
        obs, reward, terminated, truncated, info = env.step(action)

        print(
            f"  Step {step + 1:02d} | "
            f"action: {action} | "
            f"reward: {reward:.3f} | "
            f"terminated: {terminated}"
        )

        if terminated or truncated:
            obs, info = env.reset()

    env.close()
    print("\n✓ Smoke-test complete.")

✓ Reset  | obs shape : (14,)  | obs: [1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 1. 1. 1. 1.]

Running 5 random steps ...
total_waiting=0.0(float) | queue=0.0(float) | pressure=0.0(float) | collisions=0.0(float)
  Step 01 | action: 0 | reward: -0.000 | terminated: False
total_waiting=0.0(float) | queue=0.0(float) | pressure=-1.0(float) | collisions=0.0(float)
  Step 02 | action: 1 | reward: -0.300 | terminated: False
total_waiting=0.0(float) | queue=0.0(float) | pressure=-4.0(float) | collisions=0.0(float)
  Step 03 | action: 1 | reward: -1.200 | terminated: False
total_waiting=0.0(float) | queue=0.0(float) | pressure=-7.0(float) | collisions=0.0(float)
  Step 04 | action: 0 | reward: -2.100 | terminated: False
total_waiting=0.0(float) | queue=0.0(float) | pressure=-8.0(float) | collisions=0.0(float)
  Step 05 | action: 1 | reward: -2.400 | terminated: False

✓ Smoke-test complete.


SUMO_HOME: D:\SUMO
sumo binary : D:\SUMO\bin\sumo.exe

net_file   exists: True   → ..\nets\unrestricted_right_xml_gen\unrestricted_right.net.xml
route_file exists: True → ..\nets\unrestricted_right_xml_gen\routes.rou.xml
